# Contestant

A `Contestant` represents a participant in an election — either a single party or a coalition. The class is immutable; instances are created exclusively via factory methods.

In [ ]:
import pandas as pd
from ipres import Election, ElectionConfig, ConstituenciesConfig, SuperMajorityMargin, MarginUnit
from ipres.contestant import Contestant, contestantsFromParties, contestantsDictFromParties

---

## Single Party

In [ ]:
a = Contestant.from_party('A')
print(a)

# List of single-party contestants
all_contestants = contestantsFromParties(['A', 'B', 'C'])
print(all_contestants)

# Dict of single-party contestants
as_dict = contestantsDictFromParties(['A', 'B', 'C'])
print(as_dict)

In [ ]:
print(a.name)                              # 'A'
print(a.isSingleParty())                   # True
print(a.isCoalition())                     # False
print(a.members)                           # {} (empty)
print(a.getContainedParties())             # ['A']
print(a.getMemberVoteWeightsForParties())  # {'A': 1.0}

---

## Coalition

In [ ]:
a = Contestant.from_party('A')
b = Contestant.from_party('B')

# Automatic equal weighting (1/n per member)
ab = Contestant.as_coalition('A+B', [a, b])
print(ab)
print(ab.member_vote_weights)  # {'A': 0.5, 'B': 0.5}

In [ ]:
# Explicit weights: A contributed 60 %, B 40 % of coalition votes
ab_weighted = Contestant.as_coalition('A+B', [a, b], member_vote_weights={'A': 0.6, 'B': 0.4})
print(ab_weighted.member_vote_weights)
print(ab_weighted.getMemberVoteWeight('A'))  # 0.6

In [ ]:
print(ab.getMemberNames())       # ['A', 'B']
print(ab.getMemberList())        # [Contestant(party='A'), Contestant(party='B')]
print(ab.getContainedParties())  # ['A', 'B']

---

## Nested Coalition

Coalitions can be composed of other coalitions. `getMemberVoteWeightsForParties()` computes the weights of the leaf parties recursively, so the sum is always 1.0.

In [ ]:
c = Contestant.from_party('C')
ab = Contestant.as_coalition('A+B', [a, b])  # A=0.5, B=0.5

# (A+B) contributed 60 %, C 40 % of the total votes
abc = Contestant.as_coalition('A+B+C', [ab, c], member_vote_weights={'A+B': 0.6, 'C': 0.4})

print(abc.getMemberVoteWeightsForParties())
# {'A': 0.3, 'B': 0.3, 'C': 0.4}

---

## Coalition Formation During an Election

During a ballot round, a coalition can be formed via `Ballot.formCoalition()`. If it reaches the majority threshold, the election is decided — without requiring a further round.

In [ ]:
cc = ConstituenciesConfig.from_dataframe(pd.DataFrame({
    'constituency_name': ['C1'],
    'constituency_size': [100_000],
}))
config = ElectionConfig(
    constituencies_config=cc,
    participating_parties=['A', 'B', 'C', 'D'],
    parliament_majority_margin=SuperMajorityMargin(5.0, MarginUnit.PERCENT),
    seed=42,
)
election = Election(electionConfig=config)
round1 = election.start(votes={'C1': {'A': 30, 'B': 28, 'C': 25, 'D': 17}})

print("Winner after ballot?", round1.hasWinner())  # False

contestants = round1.getContestants()
round1.formCoalition("A+B", [contestants['A'], contestants['B']])

print("Winner after coalition?", election.hasWinner())  # True
print("Winner:", election.getWinner().name)             # A+B

---

## Summary

| Method / Attribute | Description |
|---|---|
| `Contestant.from_party(name)` | Create a single-party contestant |
| `Contestant.as_coalition(name, members, weights?)` | Create a coalition contestant |
| `contestantsFromParties(names)` | List of single-party contestants |
| `contestantsDictFromParties(names)` | Dict of single-party contestants |
| `.name` | Display name |
| `.isSingleParty()` / `.isCoalition()` | Check type |
| `.members` | Direct members (empty for single party) |
| `.member_vote_weights` | Vote weight fractions of direct members |
| `.getMemberList()` / `.getMemberNames()` | Member list |
| `.getContainedParties()` | All leaf parties (recursive) |
| `.getMemberVoteWeightsForParties()` | Leaf party weights (recursive, sum = 1.0) |
| `.getMemberVoteWeight(name)` | Weight of a specific direct member |
| `Ballot.formCoalition(name, members)` | Form a coalition during an election |